# Йода-LoRA: QLoRA Fine-Tuning на стиль Йоды

Дообучаем **Qwen2.5-7B** на синтетическом датасете «Йода-стиль» (русский язык).  
Используем **QLoRA** = 4-bit квантизация + LoRA адаптер + SFTTrainer (TRL).

| Шаг | Что делаем |
|-----|------------|
| 1 | Загружаем датасет `yoda_dataset.jsonl` (формат chat-messages) |
| 2 | Загружаем Qwen2.5-7B в **4-bit** (QLoRA) |
| 3 | Генерация **до** fine-tuning |
| 4 | Конфигурируем LoRA адаптер |
| 5 | Обучаем через **SFTTrainer** с chat template (≈20-30 мин на RTX 4060 Ti 16GB) |
| 6 | Генерация **после** + сохранение адаптера |

**Датасет:** 1200 пар (вопрос → ответ Йоды) в формате `{messages: [system, user, assistant]}`  
**VRAM:** ~8-9 GB модель + ~2 GB обучение = комфортно на 16 GB

In [ ]:
# datasets импортируем до torch — избегаем конфликта DLL на Windows
from datasets import Dataset as HFDataset
print('datasets ok')

import os, json, time, random
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
print('transformers + peft + trl ok')

import torch
print(f'torch: {torch.__version__}')

assert torch.cuda.is_available(), 'GPU не найден!'
gpu = torch.cuda.get_device_properties(0)
print(f'GPU: {gpu.name} | VRAM: {gpu.total_memory/1024**3:.1f} GB')

MODELS_DIR   = '../models'
DATA_PATH    = 'yoda_dataset.jsonl'
ADAPTER_PATH = f'{MODELS_DIR}/yoda_adapter'
MODEL_NAME   = 'Qwen/Qwen2.5-7B'
device       = torch.device('cuda')
os.makedirs(MODELS_DIR, exist_ok=True)

## 1. Загрузка датасета

In [ ]:
import subprocess
if not os.path.exists(DATA_PATH):
    print('Генерируем датасет...')
    subprocess.run(['python', 'generate_yoda_dataset.py'], check=True)

samples = [json.loads(l) for l in open(DATA_PATH, encoding='utf-8')]
print(f'Примеров: {len(samples)}')

random.seed(7)
print('\n--- 3 случайных примера ---')
for item in random.sample(samples, 3):
    u = item['messages'][1]['content']
    a = item['messages'][2]['content']
    print(f'\nUSER : {u}')
    print(f'YODA : {a[:120]}{"…" if len(a) > 120 else ""}')

## 2. Загрузка Qwen2.5-7B в 4-bit (QLoRA)

In [ ]:
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_compute_dtype    = torch.float16,
    bnb_4bit_use_double_quant = True,
)

print(f'Загружаем {MODEL_NAME} в 4-bit...')
print('(первый раз скачает ~15 GB — займёт время)')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_cfg,
    device_map          = 'auto',
    torch_dtype         = torch.float16,
)

n_params = sum(p.numel() for p in base_model.parameters())
print(f'Параметров: {n_params/1e9:.2f}B')
print(f'VRAM занято: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

## 3. Генерация ДО fine-tuning

Базовая модель не знает о стиле Йоды — отвечает нейтральным русским языком.

In [ ]:
SYSTEM_PROMPT = (
    'Ты — Йода, мудрый мастер джедаев из «Звёздных войн». '
    'Говори с характерной инвертированной синтаксической структурой Йоды: '
    'ставь дополнение или предикатив перед подлежащим и глаголом. '
    'Используй короткие, мудрые, созерцательные фразы на русском языке. '
    'При необходимости упоминай Силу, терпение и равновесие.'
)

TEST_QUESTIONS = [
    'Как мне стать сильнее?',
    'Что такое Сила?',
    'Как обрести душевный покой?',
    'Мне страшно.',
]


def generate(model, question, max_new_tokens=150, temperature=0.8):
    model.eval()
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': question},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            do_sample          = True,
            temperature        = temperature,
            repetition_penalty = 1.3,
            pad_token_id       = tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)


print('=' * 60)
print('  ГЕНЕРАЦИЯ ДО FINE-TUNING')
print('=' * 60)
for q in TEST_QUESTIONS:
    print(f'\n[{q}]')
    print(generate(base_model, q))

## 4. LoRA конфигурация

In [ ]:
# prepare_model_for_kbit_training — включает gradient checkpointing
# и переводит нормализации в fp32 (обязательно для QLoRA)
base_model = prepare_model_for_kbit_training(base_model)

lora_cfg = LoraConfig(
    task_type      = TaskType.CAUSAL_LM,
    r              = 16,
    lora_alpha     = 32,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout   = 0.05,
    bias           = 'none',
)

model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()
# Ожидаем: ~0.5% обучаемых из 7B — примерно 35M параметров

## 5. Подготовка датасета + обучение (SFTTrainer)

Датасет в chat-формате — применяем chat template токенайзера Qwen2.5 через `formatting_func`.  
SFTTrainer из TRL автоматически обрабатывает маскирование промпта (учим только ответ ассистента).

In [ ]:
def format_sample(sample):
    """Преобразуем messages в строку через chat template токенайзера."""
    return tokenizer.apply_chat_template(
        sample['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )

dataset  = HFDataset.from_list(samples)
split    = dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split['train']
eval_ds  = split['test']
print(f'Train: {len(train_ds)}  |  Eval: {len(eval_ds)}')

# Проверим формат одного примера
print('\nПример после chat template:')
print(format_sample(samples[0])[:300] + '...')

In [ ]:
sft_cfg = SFTConfig(
    output_dir                  = f'{MODELS_DIR}/yoda_ckpt',
    num_train_epochs            = 7,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 8,
    gradient_accumulation_steps = 8,     # эффективный батч = 16
    learning_rate               = 2e-4,
    warmup_ratio                = 0.05,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    logging_steps               = 20,
    report_to                   = 'none',
    bf16                        = True,
    fp16                        = True,
    optim                       = 'paged_adamw_8bit',
    dataloader_num_workers      = 2,
    gradient_checkpointing      = True,
    max_seq_length              = 512,
)

trainer = SFTTrainer(
    model           = model,
    args            = sft_cfg,
    train_dataset   = train_ds,
    eval_dataset    = eval_ds,
    formatting_func = format_sample,
)

t0 = time.time()
print('Начинаем QLoRA fine-tuning Qwen2.5-7B на датасете Йода...')
trainer.train()
print(f'Готово! Время: {(time.time()-t0)/60:.1f} мин')

## 6. Генерация ПОСЛЕ + сохранение адаптера

In [ ]:
print('=' * 60)
print('  СРАВНЕНИЕ: base vs QLoRA Йода')
print('=' * 60)

for q in TEST_QUESTIONS:
    print(f'\nВопрос: "{q}"')
    model.disable_adapter_layers()
    print(f'  [BASE] {generate(model, q)}')
    model.enable_adapter_layers()
    print(f'  [YODA] {generate(model, q)}')

model.enable_adapter_layers()

# Сохраняем только адаптер (~300 MB вместо 15 GB)
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f'\nАдаптер сохранён: {ADAPTER_PATH}')

meta = {
    'base_model'   : MODEL_NAME,
    'adapter_path' : ADAPTER_PATH,
    'style'        : 'Yoda speech (Russian)',
    'lora_r'       : 16,
    'quant'        : '4-bit NF4',
    'train_samples': len(train_ds),
    'test_questions': TEST_QUESTIONS,
}
with open(f'{MODELS_DIR}/yoda_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print('Сохранено: yoda_meta.json')